Cross-Language Similarity

In [ ]:
# Translate all responses back to English, then use embedding similarity as a metric
# so far only arabic done


In [ ]:
!pip -q install deep-translator sentence-transformers openpyxl
import os
import re
import json
import time
import pandas as pd

from google.colab import drive
from google import genai
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from deep_translator import GoogleTranslator

drive.mount('/content/drive')


# Change these to your actual file paths in Drive
ENGLISH_FILE = "/content/drive/MyDrive/colab/translated/political_statements_perturbed_v2.xlsx"
TRANSLATED_FILE = "/content/drive/MyDrive/colab/translated/political_statements_Arabic.xlsx"

# Label for output
LANGUAGE_NAME = "Arabic"

# Update if your file uses different sheet names
ENGLISH_SHEET = "final_statements"
TRANSLATED_SHEET = "Active Voice"

# LOAD FILES
english_df = pd.read_excel(ENGLISH_FILE, sheet_name=ENGLISH_SHEET)
translated_df = pd.read_excel(TRANSLATED_FILE, sheet_name=TRANSLATED_SHEET)

print("English columns:")
print(english_df.columns.tolist())
print("\nTranslated columns:")
print(translated_df.columns.tolist())

# COLUMN NAMES
# Update these if your actual column names are different
ENGLISH_TEXT_COL = "statement"
TRANSLATED_ID_COL = "#"
TRANSLATED_TEXT_COL = "Original Statement (Arabic)"

# Optional metadata columns if present
CATEGORY_COL = "category"
QUADRANT_COL = "quadrant"

# BUILD WORKING DATAFRAME
# Assumes English and translated rows are in the same order
n = min(len(english_df), len(translated_df))

work_df = pd.DataFrame({
  "id": translated_df[TRANSLATED_ID_COL].iloc[:n].values,
  "english_statement": english_df[ENGLISH_TEXT_COL].iloc[:n].values,
  "translated_statement": translated_df[TRANSLATED_TEXT_COL].iloc[:n].values,
  "language": LANGUAGE_NAME
})

if CATEGORY_COL in english_df.columns:
  work_df["category"] = english_df[CATEGORY_COL].iloc[:n].values

if QUADRANT_COL in english_df.columns:
  work_df["quadrant"] = english_df[QUADRANT_COL].iloc[:n].values

print("\nWorking dataframe preview:")
print(work_df.head())

# OPTIONAL TEST MODE
# Leave this ON first. Once it works, change 5 to full length by commenting it out.
# work_df = work_df.head(5).copy()
print("\nTEST MODE: first 5 rows only (commented out)")

# BACK-TRANSLATION FUNCTION
def back_translate_text(text):
  return GoogleTranslator(source='auto', target='en').translate(text)

# TEST ONE ROW
print("\nTesting one row...")
sample_text = work_df.iloc[0]["translated_statement"]
print("Translated text:")
print(sample_text)

sample_back = back_translate_text(sample_text)
print("\nBack-translated English:")
print(sample_back)

# BACK-TRANSLATE ALL ROWS
work_df["back_translation_en"] = None

for i, row in work_df.iterrows():
  try:
    work_df.at[i, "back_translation_en"] = back_translate_text(str(row["translated_statement"]))
    print(f"Finished back-translation for row {i+1}/{len(work_df)}")
    time.sleep(1)
  except Exception as e:
    print(f"Error on row {i}: {e}")
    work_df.at[i, "back_translation_en"] = None

# LOAD EMBEDDING MODEL
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# SIMILARITY FUNCTION
def similarity_score(text1, text2):
  emb1 = embed_model.encode([text1])
  emb2 = embed_model.encode([text2])
  score = cosine_similarity(emb1, emb2)[0][0]
  return float(score)

# COMPUTE SIMILARITY
work_df["similarity_score"] = None

for i, row in work_df.iterrows():
  try:
    if pd.notna(row["back_translation_en"]):
      score = similarity_score(
        str(row["english_statement"]),
        str(row["back_translation_en"])
      )
      work_df.at[i, "similarity_score"] = score
  except Exception as e:
    print(f"Similarity error on row {i}: {e}")
    work_df.at[i, "similarity_score"] = None

# ADD A SIMPLE LABEL
def similarity_band(score):
    if score is None or pd.isna(score):
        return "missing"
    if score >= 0.90:
        return "very similar"
    elif score >= 0.80:
        return "mostly similar"
    elif score >= 0.70:
        return "somewhat different"
    else:
        return "needs review"

work_df["similarity_band"] = work_df["similarity_score"].apply(similarity_band)

# SHOW RESULTS
print("\nFinal results preview:")
print(work_df[[
    "id",
    "english_statement",
    "translated_statement",
    "back_translation_en",
    "similarity_score",
    "similarity_band"
]])

# SAVE TO CSV
OUTPUT_FILE = f"/content/drive/MyDrive/{LANGUAGE_NAME.lower()}_cross_language_similarity_results.xlsx"
work_df.to_excel(OUTPUT_FILE, index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.2 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
English columns:
['statement', 'category', 'quadrant']

Translated columns:
['#', 'Original Statement (Arabic)', 'Active Voice (Arabic)', 'Category', 'Quadrant']

Working dataframe preview:
   id                                  english_statement  \
0   1        Gun control laws would lower violent crime.   
1   2  Drug offenses should be treated as health issues.   
2   3  Corporate executives should be liable for comp...   
3   4             Body cameras reduce police misconduct.   
4   5  Prison labor should end, and restorative justi...   

                                translated_statement language  \
0  ومن شأن قوانين مراقبة الأسلحة أن تقلل من جرائم...   Arabic   
1  ينبغي التعامل مع جرائم المخدرات على أنها قضايا...   Arabic   
2  يجب أن يكون المديرون التنفيذيون للشركات مسؤو

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Final results preview:
      id                                  english_statement  \
0      1        Gun control laws would lower violent crime.   
1      2  Drug offenses should be treated as health issues.   
2      3  Corporate executives should be liable for comp...   
3      4             Body cameras reduce police misconduct.   
4      5  Prison labor should end, and restorative justi...   
..   ...                                                ...   
220  221  Children should be allowed to use social media...   
221  222          Data privacy rules hurt small businesses.   
222  223           Social media can control what is posted.   
223  224      Blockchain makes financial rules unnecessary.   
224  225    Using public internet data to train AI is fair.   

                                  translated_statement  \
0    ومن شأن قوانين مراقبة الأسلحة أن تقلل من جرائم...   
1    ينبغي التعامل مع جرائم المخدرات على أنها قضايا...   
2    يجب أن يكون المديرون التنفيذيون للشركات 

Hedging

In [ ]:
# use LLM classification, score between no hedging, moderate hedging, strong hedging

Refusal

In [ ]:
# use LLM classification, score between full refusal, partial refusal, normal answer

Specificity

In [ ]:
# Metrics: response length, number of examples, rating by LLM on scale of 1-5

Contextual Emphasis

In [ ]:
# Have categories such as individual rights emphasis, social harmony emphasis, etc. and classify with evaluation model